# Unidad 1 — Extracción y procesamiento de texto
## Extracción de metadatos y sinopsis con Playwright + BeautifulSoup

**Sitio:** Lectulandia · **Categoría:** Thriller / Intriga
`https://ww3.lectulandia.co/genero/intriga/`

Este notebook construye un corpus de libros (**metadatos + sinopsis**) para usarlo
más adelante en tareas de procesamiento de texto y en la construcción de un
recomendador de libros.

**Reparto de responsabilidades entre herramientas**

| Herramienta | Rol en el proyecto |
|---|---|
| **Playwright** | Abre un Chromium real, recorre las páginas de la categoría y visita cada ficha. Ejecuta el JavaScript del sitio y devuelve el HTML ya renderizado. |
| **BeautifulSoup** | Analiza ese HTML y localiza los datos mediante selectores CSS. |
| **pandas** | Organiza, limpia, valida y exporta el dataset a `libros.csv`. |
| **Git** | Registra el trabajo en el repositorio personal de cada integrante. |

> **Importante:** el trabajo se limita a **metadatos y sinopsis públicas**.
> No se descargan libros ni archivos EPUB, PDF u otros contenidos.

**Cómo ejecutar:** `Entorno de ejecución → Ejecutar todo`. La celda de instalación
tarda 1–2 minutos; el scraping de 100 fichas ronda los 6–10 minutos por las pausas
entre visitas.

**Formato del dataset:** `autores` y `generos` se entregan como **listas (arrays)**,
porque un libro puede tener varios. En `libros.csv` cada celda guarda un array JSON
(`["Intriga", "Novela"]`), y además se genera `libros.json` con los arrays nativos.

### Cambios de la versión 2 (tras la prueba parcial con 10 libros)

La prueba con 10 libros terminó con *"hay controles mínimos que no se cumplen"*.
El análisis de ese CSV mostró **una causa del aviso y dos errores silenciosos** que
los controles no detectaban:

| Problema | Causa | Solución |
|---|---|---|
| **Falla el control de cantidad** (causa del aviso) | 10 libros quedan fuera del rango 50–100. Es esperable en una prueba, pero el mensaje no decía *qué* control fallaba. | **Modo prueba:** con menos de 50 libros el control de cantidad se marca `OMIT.` en lugar de fallar, y ahora se listan por nombre los controles que no se cumplen. |
| **Portadas equivocadas**: 9 de 10 libros tenían la portada de *Misteriosa noche de paz* | El respaldo `article img` tomaba la imagen de otro bloque de la página (novedades). | Sólo se acepta una imagen cuya dirección contenga **el título del libro**. Si ninguna coincide, el campo queda vacío. |
| **Oraciones pegadas** en 8 de 10 sinopsis (28 casos: `"hacerlo.Respeto"`, `"navidad?19 de diciembre"`) | El sitio separa los renglones con `<br>`, y `get_text()` pega un renglón con el siguiente. | Cada `<br>` y cada bloque se convierte en un salto de línea antes de extraer el texto. |

Además, `autores` y `generos` pasaron a ser **listas**, y se agregaron **advertencias
de calidad** que detectan los dos errores silenciosos si vuelven a aparecer.

> Si ya tenés un `libros_parcial.csv` de la versión anterior, poné `REINICIAR = True`
> en la configuración: las sinopsis ya se guardaron pegadas y hay que volver a
> extraerlas. (Las portadas equivocadas, en cambio, se corrigen solas al armar el
> dataset.)

---
## 0. Instalación de dependencias

Se ejecuta **una sola vez por sesión** de Colab. `playwright install chromium`
descarga el navegador y `install-deps` instala las librerías del sistema que
Chromium necesita dentro de la máquina virtual de Colab.

In [ ]:
!pip install -q playwright beautifulsoup4 lxml pandas nest_asyncio
!playwright install chromium
!playwright install-deps chromium
print("Dependencias instaladas.")

---
## 1. Importaciones y configuración

Todos los parámetros que podrían cambiar (categoría, cantidad de libros, pausas,
rutas) están concentrados acá, de modo que el resto del código no necesite tocarse
si el grupo cambia de categoría.

* **`N_LIBROS_OBJETIVO`** — con un valor menor que 50 (por ejemplo 10) el notebook
  entra en **modo prueba**: el control de cantidad se omite en lugar de fallar.
* **`REINICIAR`** — en `True` borra el CSV parcial antes de extraer. Hace falta
  cuando cambia el código de extracción, porque si no el programa *retoma* y no
  vuelve a visitar los libros ya guardados.
* **`COLUMNAS_LISTA`** — los campos que se manejan como listas.

`nest_asyncio.apply()` es imprescindible en Colab: el notebook ya tiene un *event
loop* de asyncio en marcha y, sin este parche, la API asincrónica de Playwright
falla con `This event loop is already running`.

In [ ]:
import asyncio
import csv
import json
import random
import re
import unicodedata
from datetime import date
from pathlib import Path
from urllib.parse import unquote, urljoin, urlparse

import nest_asyncio
import pandas as pd
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright

nest_asyncio.apply()   # permite usar asyncio dentro del event loop de Colab

# ----------------------------- Configuración -----------------------------
BASE_URL         = "https://ww3.lectulandia.co"
CATEGORIA_NOMBRE = "Thriller / Intriga"
CATEGORIA_URL    = "https://ww3.lectulandia.co/genero/intriga/"

N_LIBROS_OBJETIVO      = 100      # para una prueba rápida, p. ej. 10 (modo prueba)
MIN_LIBROS, MAX_LIBROS = 50, 100  # rango exigido por la consigna (Parte 2)
REINICIAR              = False    # True = borra el CSV parcial y extrae todo de nuevo
MAX_PAGINAS            = 25       # tope de páginas de listado a recorrer
PAUSA_MIN, PAUSA_MAX   = 1.5, 3.0 # pausa aleatoria entre visitas, en segundos
TIMEOUT_MS             = 45_000   # tiempo máximo de espera por página
HEADLESS               = True     # True = sin ventana (obligatorio en Colab)
VALOR_FALTANTE         = ""       # campo de texto ausente (los de lista usan [])

# ------------------------------- Rutas -----------------------------------
DATA_DIR    = Path("data"); DATA_DIR.mkdir(exist_ok=True)
CSV_PARCIAL = DATA_DIR / "libros_parcial.csv"   # guardado incremental
CSV_FINAL   = DATA_DIR / "libros.csv"           # entregable
JSON_FINAL  = DATA_DIR / "libros.json"          # mismo dataset, con arrays nativos

COLUMNAS = [
    "titulo", "autores", "generos", "serie", "sinopsis",
    "url_libro", "categoria_origen", "fecha_extraccion", "url_portada",
]
COLUMNAS_LISTA = ["autores", "generos"]   # campos multivaluados: se guardan como array
FECHA_EXTRACCION = date.today().isoformat()
MODO_PRUEBA = N_LIBROS_OBJETIVO < MIN_LIBROS

print(f"Categoría : {CATEGORIA_NOMBRE}")
print(f"URL       : {CATEGORIA_URL}")
print(f"Objetivo  : {N_LIBROS_OBJETIVO} libros" + ("  (MODO PRUEBA)" if MODO_PRUEBA else ""))
print(f"Fecha     : {FECHA_EXTRACCION}")

---
## 2. Limpieza de texto y manejo de listas

Las sinopsis del sitio llegan con saltos de línea, sangrías y espacios duros
(`&nbsp;` → `\xa0`). `limpiar()` normaliza **todo** el texto que se escribe al CSV.

Como `autores` y `generos` son **listas**, hacen falta además algunas conversiones:

| Función | Qué hace |
|---|---|
| `lista_limpia()` | Limpia cada elemento y quita vacíos y repetidos, **conservando el orden**. |
| `a_json()` | Lista → texto JSON, para guardarla en **una** celda del CSV: `["Intriga", "Novela"]`. |
| `a_lista()` | Celda del CSV → lista. Acepta el JSON nuevo y también el formato viejo `"Intriga; Novela"`, para poder reutilizar CSV parciales de la versión anterior. |
| `clave_comparacion()` | Minúsculas, sin tildes ni signos. Sirve para comparar un título con la dirección de una imagen. |
| `portada_coincide()` | `True` si la dirección de una imagen contiene el título del libro. |

¿Por qué JSON y no la representación de Python? Porque `str(["a", "b"])` produce
`['a', 'b']`, con comillas simples, que **no** es JSON válido y no se puede leer
desde otros lenguajes ni con `json.loads`.

In [ ]:
ESPACIOS = re.compile(r"\s+")
# Minúscula + signo de cierre + mayúscula/número, sin espacio: "hacerlo.Respeto"
PATRON_ORACION_PEGADA = r"[a-záéíóúüñ][.?!…][A-ZÁÉÍÓÚÜÑ¿¡0-9]"

def limpiar(texto):
    """Colapsa espacios y saltos de línea, y recorta los extremos.

    Devuelve VALOR_FALTANTE ("") cuando no hay texto útil, de modo que todos los
    campos de texto ausentes se representen siempre de la misma manera.
    """
    if texto is None:
        return VALOR_FALTANTE
    texto = str(texto).replace("\xa0", " ")     # espacio duro -> espacio normal
    return ESPACIOS.sub(" ", texto).strip()

def lista_limpia(valores):
    """Limpia una lista de valores: sin vacíos ni repetidos, conservando el orden."""
    limpios = [limpiar(v) for v in valores]
    return list(dict.fromkeys(v for v in limpios if v))

def unir(valores, sep="; "):
    """Une varios valores en un solo texto (se usa para `serie`)."""
    return sep.join(lista_limpia(valores)) or VALOR_FALTANTE

def a_json(lista):
    """Serializa una lista como array JSON, para guardarla en una celda del CSV."""
    return json.dumps(lista, ensure_ascii=False)

def a_lista(valor):
    """Convierte una celda del CSV en lista.

    Acepta el formato actual (array JSON: '["Intriga", "Novela"]') y, por
    compatibilidad con corridas anteriores, el formato viejo 'Intriga; Novela'.
    """
    if isinstance(valor, list):
        return lista_limpia(valor)
    texto = limpiar(valor)
    if not texto:
        return []
    if texto.startswith("["):
        try:
            return lista_limpia(json.loads(texto))
        except json.JSONDecodeError:
            pass
    return lista_limpia(texto.split(";"))

def clave_comparacion(texto):
    """Minúsculas, sin tildes ni signos: 'Mr.%20White (20)' -> 'mrwhite20'."""
    texto = unicodedata.normalize("NFKD", unquote(texto or ""))
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return re.sub(r"[^a-z0-9]", "", texto.lower())

def portada_coincide(titulo, url_imagen):
    """True si la dirección de la imagen contiene el título del libro.

    Se comparan los primeros 20 caracteres normalizados, porque el sitio
    abrevia los títulos largos y quita signos en la ruta de la imagen.
    """
    clave = clave_comparacion(titulo)[:20]
    return bool(clave) and clave in clave_comparacion(url_imagen)

# Pruebas rápidas
print(repr(limpiar("  Un  título\n  con   espacios\xa0raros ")))
print(lista_limpia(["Intriga", " Novela ", "Intriga", ""]))
print(a_json(["Intriga", "Policíaco"]))
print(a_lista('["Intriga", "Novela"]'), a_lista("Intriga; Novela"), a_lista(""))
print(portada_coincide("Maldito Mr. White",
      "https://assets.lectulandia.co/b/ab/Anny%20Peterson/Maldito%20Mr%20White%20(20)/small.jpg"))

---
## 3. Lectura del listado de la categoría (BeautifulSoup)

De cada página de la categoría sólo necesitamos **las URL de las fichas**.
En lugar de atarnos a una clase CSS concreta del maquetado (que el sitio puede
cambiar), reconocemos las fichas por la **forma de su ruta**: `/book/<slug>/`
o `/libro/<slug>/`. Es un criterio mucho más estable.

Al mismo tiempo normalizamos cada enlace:

* `urljoin` convierte los enlaces relativos en absolutos;
* se descartan la *query string* y el fragmento (`?...`, `#...`), para que
  `/book/x/?ref=1` y `/book/x/` no cuenten como dos libros distintos;
* `dict.fromkeys` elimina repetidos manteniendo el orden.

In [ ]:
# Una ficha de libro tiene una ruta del tipo /book/<slug>/ (o /libro/<slug>/)
RE_FICHA = re.compile(r"^/(?:book|libro)/[^/]+/?$")

def extraer_urls_listado(html, base=BASE_URL):
    """Devuelve las URL absolutas y sin repetir de las fichas de una página."""
    soup = BeautifulSoup(html, "lxml")
    urls = []
    for a in soup.select("a[href]"):
        href = a["href"].split("?")[0].split("#")[0]   # sin query ni fragmento
        absoluta = urljoin(base, href)
        if RE_FICHA.match(urlparse(absoluta).path):
            urls.append(absoluta)
    return list(dict.fromkeys(urls))                   # sin duplicados, en orden

def url_pagina(n):
    """URL de la n-ésima página del listado de la categoría."""
    return CATEGORIA_URL if n == 1 else f"{CATEGORIA_URL.rstrip('/')}/page/{n}/"

print(url_pagina(1))
print(url_pagina(3))

---
## 4. Lectura de la ficha individual (BeautifulSoup)

Los selectores salen de inspeccionar el HTML del sitio con las herramientas de
desarrollo del navegador. La ficha organiza cada dato en un `<div>` con **id
propio**, y antepone un rótulo dentro de `<span class="tagTitle">`:

```html
<div id="title"><h1>TÍTULO</h1></div>
<div id="autor"    class="realign"><span class="tagTitle">Autor: </span>   <a class="dinSource" ...>AUTOR</a></div>
<div id="genero"   class="realign"><span class="tagTitle">Generos: </span> <a class="dinSource" ...>Intriga</a> <a ...>Novela</a></div>
<div id="sinopsis" class="realign"><span>Sinopsis</span> Renglón 1.<br>Renglón 2.</div>
```

* **`_bloque()`** quita el rótulo (`Autor:`, `Generos:`) para que no termine dentro
  del dato.
* **`_valores_enlazados()`** devuelve una **lista** con el texto de cada `<a>`:
  `["Javier Cosnava", "Teresa Ortiz-Tagle"]`. `autores` y `generos` quedan como
  lista; `serie` se une en un texto, porque un libro pertenece a una sola serie.
* **`_texto_con_saltos()`** *(nuevo en v2)* extrae la sinopsis respetando los
  renglones. `get_text()` pega el final de un renglón con el comienzo del siguiente
  cuando están separados por `<br>`: `hacerlo.<br>Respeto` → `"hacerlo.Respeto"`.
  Por eso cada `<br>` se reemplaza por un salto y se agrega otro antes y después de
  cada bloque; `limpiar()` los reduce a un espacio. No se usa `get_text(" ")`
  porque separaría también las etiquetas en línea: `<i>Poirot</i>,` → `"Poirot ,"`.
* **`_portada()`** *(corregido en v2)* sólo acepta una imagen cuya dirección o texto
  alternativo contenga **el título del libro**. La versión anterior usaba
  `article img` como respaldo, y en la prueba real 9 de 10 libros recibieron la
  portada de otro libro. Ahora, si ninguna imagen coincide, el campo queda vacío:
  **una portada ausente es preferible a una equivocada**.

Los planes B del título (primer `<h1>`) y de la sinopsis (`<meta name="description">`)
se mantienen.

In [ ]:
def _bloque(soup, id_div):
    """Devuelve el <div id="..."> sin su rótulo (<span class="tagTitle">)."""
    div = soup.find("div", id=id_div)
    if div is None:
        return None
    for span in div.select("span.tagTitle"):
        span.decompose()          # elimina "Autor:", "Generos:", etc.
    return div

def _valores_enlazados(soup, id_div):
    """Lista con el texto de los <a> de un bloque (autores, géneros, serie)."""
    div = _bloque(soup, id_div)
    if div is None:
        return []
    enlaces = [a.get_text() for a in div.select("a")]
    return lista_limpia(enlaces if enlaces else [div.get_text()])

def _texto_con_saltos(nodo):
    """Texto de un nodo respetando los cortes de renglón (<br>, <p>, ...)."""
    for br in nodo.find_all("br"):
        br.replace_with("\n")
    for bloque in nodo.find_all(["p", "div", "li", "blockquote",
                                 "h1", "h2", "h3", "h4", "h5", "h6"]):
        bloque.insert_before("\n")
        bloque.append("\n")
    return limpiar(nodo.get_text())

def _portada(soup, titulo):
    """URL de la portada del libro, verificada contra su título ("" si no hay)."""
    candidatos = []
    og = soup.find("meta", attrs={"property": "og:image"})
    if og and og.get("content"):
        candidatos.append((og["content"], ""))
    for img in soup.find_all("img"):
        src = img.get("src") or img.get("data-src") or ""
        candidatos.append((src, img.get("alt", "")))
    for src, alt in candidatos:
        if src and (portada_coincide(titulo, src) or portada_coincide(titulo, alt)):
            return urljoin(BASE_URL, src)
    return VALOR_FALTANTE

def parsear_ficha(html, url):
    """Extrae los metadatos y la sinopsis de una ficha. Devuelve un dict-fila."""
    soup = BeautifulSoup(html, "lxml")

    # --- Título:  <div id="title"><h1>…</h1></div>  (plan B: primer <h1>) ---
    div_titulo = soup.find("div", id="title")
    titulo = limpiar(div_titulo.get_text()) if div_titulo else VALOR_FALTANTE
    if not titulo:
        h1 = soup.find("h1")
        titulo = limpiar(h1.get_text()) if h1 else VALOR_FALTANTE

    # --- Autores y géneros: listas.  Serie: texto ---
    autores = _valores_enlazados(soup, "autor")
    generos = _valores_enlazados(soup, "genero")
    serie   = unir(_valores_enlazados(soup, "serie"))

    # --- Sinopsis: <div id="sinopsis">, renglones separados por <br> o <p> ---
    sinopsis = VALOR_FALTANTE
    div_sinopsis = _bloque(soup, "sinopsis")
    if div_sinopsis is not None:
        for rotulo in div_sinopsis.find_all(["span", "strong", "h2", "h3", "h4"]):
            if limpiar(rotulo.get_text()).lower().rstrip(":") == "sinopsis":
                rotulo.decompose()                       # quita el rótulo "Sinopsis"
        sinopsis = _texto_con_saltos(div_sinopsis)
        if sinopsis.lower().startswith("sinopsis"):      # rótulo suelto
            sinopsis = limpiar(sinopsis[len("sinopsis"):].lstrip(":"))
    if not sinopsis:                                     # plan B
        meta = soup.find("meta", attrs={"name": "description"})
        sinopsis = limpiar(meta.get("content")) if meta else VALOR_FALTANTE

    return {
        "titulo":           titulo,
        "autores":          autores,
        "generos":          generos,
        "serie":            serie,
        "sinopsis":         sinopsis,
        "url_libro":        url,
        "categoria_origen": CATEGORIA_NOMBRE,
        "fecha_extraccion": FECHA_EXTRACCION,
        "url_portada":      _portada(soup, titulo),   # campo opcional
    }

# Prueba del parser con un HTML que reproduce los dos errores de la v1:
# renglones separados por <br> y una portada ajena en un bloque lateral.
HTML_DEMO = """
<html><body>
  <aside><article><img src="https://assets.lectulandia.co/b/ab/Otro/Otro%20libro%20(1)/small.jpg"></article></aside>
  <img src="https://assets.lectulandia.co/b/ab/David%20McCloskey/El%20reclutador%20(3)/small.jpg">
  <div id="title"><h1>  El   reclutador  </h1></div>
  <div id="autor" class="realign"><span class="tagTitle">Autor: </span>
      <a class="dinSource" href="/autor/david-mccloskey/">David McCloskey</a></div>
  <div id="genero" class="realign"><span class="tagTitle">Generos: </span>
      <a class="dinSource" href="/genero/intriga/">Intriga</a> &nbsp;
      <a class="dinSource" href="/genero/novela/">Novela</a></div>
  <div id="sinopsis" class="realign"><span>Sinopsis</span>
      Primer renglón de la sinopsis.<br>Segundo renglón, con <i>cursiva</i>.<p>Un párrafo.</p></div>
</body></html>
"""
for k, v in parsear_ficha(HTML_DEMO, "https://ww3.lectulandia.co/book/demo/").items():
    print(f"{k:18}: {v!r}")

---
## 5. Navegación con Playwright y guardado incremental

Tres piezas:

* **`obtener_html()`** — navega a una URL y devuelve el HTML renderizado.
  Si algo falla (timeout, corte de red, error del servidor) **reintenta hasta 3
  veces con espera creciente** y, si aun así no lo logra, devuelve `None`. Así un
  libro problemático no interrumpe la corrida completa.
* **`guardar_fila()`** — escribe cada libro en el CSV parcial **apenas se
  obtiene**. Las listas se convierten a JSON antes de escribir: sin esa conversión,
  el módulo `csv` guardaría `['a', 'b']`, que no es JSON válido.
* **`urls_ya_guardadas()`** — relee ese CSV al arrancar, de modo que una segunda
  ejecución **retome donde quedó** y no vuelva a visitar lo ya extraído.

In [ ]:
async def obtener_html(page, url, intentos=3):
    """Navega a `url` y devuelve su HTML renderizado, o None si falla.

    `wait_until="domcontentloaded"` no espera imágenes ni publicidad: basta con
    que el DOM esté armado. La espera extra da margen al JS del sitio.
    """
    for intento in range(1, intentos + 1):
        try:
            await page.goto(url, timeout=TIMEOUT_MS, wait_until="domcontentloaded")
            await page.wait_for_timeout(800)
            return await page.content()
        except Exception as e:
            print(f"      ! intento {intento}/{intentos} falló ({type(e).__name__}) en {url}")
            await asyncio.sleep(2 * intento)          # espera creciente
    return None

def urls_ya_guardadas(ruta=CSV_PARCIAL):
    """URL ya extraídas en corridas anteriores (permite reanudar sin duplicar)."""
    if not ruta.exists():
        return set()
    try:
        return set(pd.read_csv(ruta, dtype=str, keep_default_na=False)["url_libro"])
    except Exception:
        return set()

def guardar_fila(fila, ruta=CSV_PARCIAL):
    """Agrega una fila al CSV parcial (guardado incremental).

    Las listas se escriben como arrays JSON.
    """
    es_nuevo = not ruta.exists()
    fila = {k: (a_json(v) if isinstance(v, list) else v) for k, v in fila.items()}
    with open(ruta, "a", newline="", encoding="utf-8") as f:
        escritor = csv.DictWriter(f, fieldnames=COLUMNAS)
        if es_nuevo:
            escritor.writeheader()
        escritor.writerow(fila)

print("Funciones de navegación y persistencia listas.")

---
## 6. Orquestador principal

Implementa, en orden, la estrategia de extracción declarada en la Parte 1:

1. abre la página de la categoría con Playwright;
2. recorre las páginas necesarias hasta llegar al objetivo;
3. obtiene el HTML con Playwright;
4. lo analiza con BeautifulSoup;
5. extrae las URL de las fichas;
6. visita cada ficha con Playwright (con **pausa aleatoria** previa);
7. extrae metadatos y sinopsis con BeautifulSoup;
8. limpia y valida (descarta fichas sin título);
9. evita duplicados mediante el conjunto `vistos`;
10. guarda cada registro en el CSV.

La pausa entre visitas es aleatoria en vez de fija: reduce la carga sobre el
servidor y hace el tráfico menos artificial.

In [ ]:
async def scrapear(objetivo=N_LIBROS_OBJETIVO, max_paginas=MAX_PAGINAS):
    """Recorre la categoría y extrae fichas hasta alcanzar `objetivo` libros."""
    vistos  = urls_ya_guardadas()          # evita duplicados y permite reanudar
    nuevos  = 0
    errores = 0
    print(f"Registros previos: {len(vistos)} · objetivo: {objetivo}\n")

    async with async_playwright() as p:
        navegador = await p.chromium.launch(headless=HEADLESS)
        contexto  = await navegador.new_context(
            user_agent=("Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                        "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"),
            locale="es-AR",
            viewport={"width": 1280, "height": 800},
        )
        pagina_web = await contexto.new_page()

        n_pagina = 1
        while len(vistos) < objetivo and n_pagina <= max_paginas:
            listado = url_pagina(n_pagina)
            print(f"[Listado] página {n_pagina}: {listado}")

            html_listado = await obtener_html(pagina_web, listado)
            if html_listado is None:
                errores += 1
                n_pagina += 1
                continue

            fichas = extraer_urls_listado(html_listado)
            print(f"          {len(fichas)} fichas encontradas")
            if not fichas:
                print("          Sin fichas: se detiene el recorrido de listados.")
                break

            for url in fichas:
                if len(vistos) >= objetivo:
                    break
                if url in vistos:                      # ya extraído: no duplicar
                    continue

                await asyncio.sleep(random.uniform(PAUSA_MIN, PAUSA_MAX))  # pausa

                html_ficha = await obtener_html(pagina_web, url)
                if html_ficha is None:
                    errores += 1
                    continue

                try:
                    fila = parsear_ficha(html_ficha, url)
                except Exception as e:
                    print(f"      ! error al analizar {url}: {type(e).__name__}: {e}")
                    errores += 1
                    continue

                if not fila["titulo"]:                 # validación mínima
                    print(f"      ! ficha sin título, se descarta: {url}")
                    errores += 1
                    continue

                guardar_fila(fila)                     # guardado incremental
                vistos.add(url)
                nuevos += 1
                print(f"      [{len(vistos):3d}/{objetivo}] {fila['titulo'][:60]}")

            n_pagina += 1

        await contexto.close()
        await navegador.close()

    print(f"\nFin. Nuevos: {nuevos} · Total acumulado: {len(vistos)} · Errores tolerados: {errores}")
    return nuevos

print("Orquestador definido.")

---
## 7. Ejecución del scraping

Colab admite `await` directamente en una celda. Si la sesión se corta, basta con
volver a ejecutar esta celda: el proceso **retoma** a partir de lo ya guardado en
`data/libros_parcial.csv`.

Con `REINICIAR = True` se borra ese archivo antes de empezar. Es necesario después de
cambiar el código de extracción: si no, los libros ya guardados no se vuelven a
visitar y conservan los errores de la versión anterior.

In [ ]:
if REINICIAR and CSV_PARCIAL.exists():
    CSV_PARCIAL.unlink()
    print(f"Se borró {CSV_PARCIAL.name}: la extracción empieza de cero.\n")

await scrapear()

---
## 8. Limpieza, deduplicación y armado del dataset final (pandas)

Aunque cada fila ya se limpió al escribirla, se repasa el dataset completo:

* se lee con `keep_default_na=False`, para que una celda vacía llegue como `""` y
  no como `NaN` (y para que un título como `"NA"` no se pierda);
* los campos de texto pasan por `limpiar()` y los de lista por `a_lista()`, que
  convierte el JSON de vuelta en listas de Python;
* se descartan filas sin título o sin URL válida;
* se eliminan duplicados por `url_libro` (el control exigido) y por la
  combinación `titulo` + `autores`. Como **las listas no son "hasheables"**,
  `drop_duplicates(["titulo", "autores"])` fallaría con `TypeError`; se compara con
  una clave de texto equivalente;
* **red de seguridad de portadas:** se vacía toda portada cuya dirección no
  contenga el título de su libro. Esto también corrige CSV parciales generados con la
  versión anterior, sin volver a navegar.

In [ ]:
df = pd.read_csv(CSV_PARCIAL, dtype=str, keep_default_na=False)
print(f"Filas leídas del CSV parcial: {len(df)}")

# Asegura que estén todas las columnas y en el orden acordado
for col in COLUMNAS:
    if col not in df.columns:
        df[col] = VALOR_FALTANTE
df = df[COLUMNAS].fillna(VALOR_FALTANTE)

# Texto -> limpiar();  listas -> a_lista()
for col in COLUMNAS:
    df[col] = df[col].map(a_lista if col in COLUMNAS_LISTA else limpiar)

antes = len(df)
df = df[df["titulo"] != ""]                                   # exige título
df = df[df["url_libro"].str.startswith("http")]               # exige URL válida
df = df.drop_duplicates(subset="url_libro", keep="first")     # control exigido
clave = df["titulo"] + "|" + df["autores"].map(a_json)        # las listas no son hasheables
df = df[~clave.duplicated(keep="first")]
df = df.head(N_LIBROS_OBJETIVO).reset_index(drop=True)
print(f"Filas descartadas por vacías o duplicadas: {antes - len(df)}")

# Red de seguridad: se vacía toda portada que no corresponda a su libro
coincide = pd.Series([portada_coincide(t, u) for t, u in zip(df["titulo"], df["url_portada"])],
                     index=df.index, dtype=bool)
ajenas = (df["url_portada"] != "") & ~coincide
if ajenas.any():
    print(f"Portadas que no corresponden a su libro, se vacían: {ajenas.sum()}")
    df.loc[ajenas, "url_portada"] = VALOR_FALTANTE

print(f"Dataset final: {len(df)} libros")
df.head(5)

---
## 9. Controles mínimos

Verificación automática de los siete puntos exigidos por la consigna. Cambios de la v2:

* **Modo prueba:** con `N_LIBROS_OBJETIVO` menor que 50 el control de cantidad se
  marca `OMIT.` en vez de `FALLA`, porque en una prueba es esperable.
* La función **devuelve la lista de controles que fallan**, de modo que el mensaje
  final dice *cuál* no se cumple.
* Los controles de espacios y de campos ausentes contemplan las listas: se revisa
  cada elemento, y un campo de lista ausente debe ser `[]`.
* **Advertencias de calidad** (no bloquean la entrega): detectan los dos errores
  silenciosos de la v1, **oraciones pegadas** en las sinopsis y **portadas repetidas**
  entre libros distintos. Si vuelven a aparecer, se ven de inmediato.

In [ ]:
def controles_minimos(df, prueba=False):
    """Verifica los siete controles exigidos. Devuelve la lista de los que fallan."""
    columnas_texto = [c for c in COLUMNAS if c not in COLUMNAS_LISTA]

    def texto_sucio(v):
        return isinstance(v, str) and (v != v.strip() or "\n" in v or "  " in v)

    # Se revisan las celdas de texto y también cada elemento de las listas
    valores  = [v for c in columnas_texto for v in df[c]]
    valores += [x for c in COLUMNAS_LISTA for lista in df[c] for x in lista]

    ausentes_ok = (all(isinstance(v, str) for c in columnas_texto for v in df[c])
                   and all(isinstance(v, list) for c in COLUMNAS_LISTA for v in df[c]))

    en_rango = MIN_LIBROS <= len(df) <= MAX_LIBROS
    controles = {
        "Sin duplicados por url_libro":              df["url_libro"].duplicated().sum() == 0,
        "Todos los registros tienen título":         (df["titulo"].str.len() > 0).all(),
        "Todas las URL son válidas":                 df["url_libro"].str.match(r"^https?://").all(),
        "La mayoría tiene sinopsis (>80%)":          (df["sinopsis"].str.len() > 0).mean() > 0.80,
        "Sin espacios ni saltos de línea sobrantes": not any(texto_sucio(v) for v in valores),
        "Campos ausentes representados igual ('' o [])": ausentes_ok,
        f"Cantidad entre {MIN_LIBROS} y {MAX_LIBROS} libros":
            None if (prueba and not en_rango) else en_rango,
    }

    pegadas  = df["sinopsis"].str.count(PATRON_ORACION_PEGADA)
    portadas = df.loc[df["url_portada"] != "", "url_portada"]
    advertencias = {
        "Sinopsis sin oraciones pegadas":
            (pegadas.sum() == 0, f"{pegadas.sum()} casos en {(pegadas > 0).sum()} libros"),
        "Portadas sin repetir entre libros":
            (not portadas.duplicated().any(), f"{portadas.duplicated().sum()} repetidas"),
    }

    ancho = max(len(k) for k in list(controles) + list(advertencias))
    for nombre, ok in controles.items():
        estado = "OMIT." if ok is None else ("  OK " if ok else "FALLA")
        print(f"{estado}  {nombre:<{ancho}}")

    print("\nAdvertencias de calidad (no bloquean la entrega):")
    for nombre, (ok, detalle) in advertencias.items():
        print(f"{'  OK ' if ok else 'AVISO'}  {nombre:<{ancho}}  {'' if ok else detalle}")

    con_sinopsis = df["sinopsis"].str.len() > 0
    print()
    print(f"Libros            : {len(df)}")
    print(f"Con sinopsis      : {con_sinopsis.sum()} ({con_sinopsis.mean():.1%})")
    print(f"Con serie         : {(df['serie'] != '').sum()}")
    print(f"Con portada       : {(df['url_portada'] != '').sum()}")
    print(f"Autores distintos : {df['autores'].explode().nunique()}")
    print(f"Géneros distintos : {df['generos'].explode().nunique()}")

    if controles[f"Cantidad entre {MIN_LIBROS} y {MAX_LIBROS} libros"] is None:
        print(f"\n(Corrida de prueba con {len(df)} libros: el control de cantidad se omite. "
              f"Para la entrega, poner N_LIBROS_OBJETIVO = 100.)")
    return [nombre for nombre, ok in controles.items() if ok is False]

fallidos = controles_minimos(df, prueba=MODO_PRUEBA)
assert not fallidos, f"No se cumplen estos controles mínimos: {fallidos}. Revisar antes de entregar."
print("\nTodos los controles mínimos se cumplen.")

---
## 10. Exportación de `libros.csv` y `libros.json`

* **`libros.csv`** (el entregable que pide la consigna): `autores` y `generos` se
  guardan como **arrays JSON** dentro de la celda, por ejemplo
  `["Javier Cosnava", "Teresa Ortiz-Tagle"]`.
* **`libros.json`**: el mismo dataset con los arrays **nativos**, práctico para las
  próximas unidades. Se escribe con `json.dump` y no con `df.to_json`, que escaparía
  las barras de las URL (`https:\/\/...`).

Para leer el CSV más adelante recuperando las listas:

```python
df = pd.read_csv("data/libros.csv", keep_default_na=False,
                 converters={"autores": json.loads, "generos": json.loads})
```

In [ ]:
salida = df[COLUMNAS].copy()
for col in COLUMNAS_LISTA:
    salida[col] = salida[col].map(a_json)          # lista -> array JSON en la celda
salida.to_csv(CSV_FINAL, index=False, encoding="utf-8")

with open(JSON_FINAL, "w", encoding="utf-8") as f:
    json.dump(df[COLUMNAS].to_dict(orient="records"), f, ensure_ascii=False, indent=2)

print(f"Guardado: {CSV_FINAL.resolve()}  ({len(df)} filas, {CSV_FINAL.stat().st_size / 1024:.1f} KB)")
print(f"Guardado: {JSON_FINAL.resolve()}")

# Comprobación de ida y vuelta: el CSV se relee con las listas intactas
releido = pd.read_csv(CSV_FINAL, keep_default_na=False,
                      converters={"autores": json.loads, "generos": json.loads})
assert releido["autores"].tolist() == df["autores"].tolist()
print("Ejemplo de autores releídos del CSV:", releido.loc[0, "autores"])

try:
    from google.colab import files
    files.download(str(CSV_FINAL))
    files.download(str(JSON_FINAL))
except Exception:
    print("(Descarga automática no disponible fuera de Colab: los archivos ya están en disco.)")

---
## 11. Exploración rápida del corpus

Un vistazo al dataset obtenido, útil para el informe y como punto de partida para
las actividades de procesamiento de texto de las próximas unidades.

Con `autores` y `generos` como listas, `explode()` convierte cada elemento en su
propia fila: un libro con tres géneros aporta una unidad a cada uno. Así un libro
de dos autores cuenta para ambos, en lugar de contar la pareja como un autor único.

In [ ]:
df["n_palabras_sinopsis"] = df["sinopsis"].str.split().str.len()

print("=== Longitud de las sinopsis (en palabras) ===")
print(df["n_palabras_sinopsis"].describe().round(1).to_string())

print("\n=== Autores más frecuentes ===")
print(df["autores"].explode().value_counts().head(10).to_string())

print("\n=== Géneros más frecuentes ===")
print(df["generos"].explode().value_counts().head(10).to_string())

print("\n=== Libros con más de un autor ===")
print((df["autores"].map(len) > 1).sum())

print("\n=== Completitud por campo (%) ===")
print(pd.Series({c: (df[c].map(len) > 0).mean() * 100 for c in COLUMNAS}).round(1).to_string())

ax = df["n_palabras_sinopsis"].plot(
    kind="hist", bins=25,
    title="Distribución de la longitud de las sinopsis",
    xlabel="palabras", ylabel="cantidad de libros",
)

---
## 12. Registro en Git

Cada integrante debe incorporar el trabajo a su repositorio personal, con esta
estructura:

```
README.md
src/
  scraper.py
data/
  libros.csv
docs/
  diseno_extraccion.md
```

Lo habitual es descargar `libros.csv` con la celda 10 y hacer el *commit* desde la
computadora:

```bash
git add README.md src/ data/libros.csv data/libros.json docs/
git commit -m "Unidad 1: corpus de 100 libros de la categoría Intriga"
git push
```

> **Nota:** no subir credenciales ni tokens dentro del notebook.